# Neural Network From Scratch — XOR Problem (SOLUTION)

Complete working implementation using only NumPy.

## Step 1: Import Libraries

In [ ]:
import numpy as np

## Step 2: Define the Dataset

In [ ]:
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float64)
y = np.array([[0],[1],[1],[0]], dtype=np.float64)
print('X:', X)
print('y:', y)

## Step 3: Initialize Weights and Biases

Network: 2 inputs → 4 hidden neurons → 1 output

In [ ]:
np.random.seed(42)
W1 = np.random.randn(2, 4)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1)
b2 = np.zeros((1, 1))
print('W1:', W1.shape, ' b1:', b1.shape)
print('W2:', W2.shape, ' b2:', b2.shape)

## Step 4: Activation Functions

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return sigmoid(x) * (1 - sigmoid(x))

print(sigmoid(0))             # 0.5
print(sigmoid_derivative(0))  # 0.25

## Step 5: Forward Pass

In [ ]:
def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    return z1, a1, z2, a2

z1, a1, z2, a2 = forward(X, W1, b1, W2, b2)
print('a2 shape:', a2.shape)
print('initial predictions:', a2)

## Step 6: Compute Loss (Binary Cross-Entropy)

In [ ]:
def compute_loss(y, a2):
    return -np.mean(y * np.log(a2 + 1e-8) + (1 - y) * np.log(1 - a2 + 1e-8))

print('Initial loss:', compute_loss(y, a2))

## Step 7: Backward Pass (Backpropagation)

In [ ]:
def backward(X, y, z1, a1, z2, a2, W1, W2):
    n = X.shape[0]
    dL_da2 = -(y / (a2 + 1e-8) - (1 - y) / (1 - a2 + 1e-8)) / n
    dL_dz2 = dL_da2 * sigmoid_derivative(z2)
    dW2 = a1.T @ dL_dz2
    db2 = np.sum(dL_dz2, axis=0, keepdims=True)
    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * sigmoid_derivative(z1)
    dW1 = X.T @ dL_dz1
    db1 = np.sum(dL_dz1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2

## Step 8: Parameter Update

In [ ]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, lr=0.1):
    return W1 - lr*dW1, b1 - lr*db1, W2 - lr*dW2, b2 - lr*db2

## Step 9: Training Loop

In [ ]:
losses = []
for epoch in range(10000):
    z1, a1, z2, a2 = forward(X, W1, b1, W2, b2)
    loss = compute_loss(y, a2)
    losses.append(loss)
    dW1, db1, dW2, db2 = backward(X, y, z1, a1, z2, a2, W1, W2)
    W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2)
    if epoch % 1000 == 0:
        print(f'Epoch {epoch:5d} | Loss: {loss:.4f}')

## Step 10: Evaluate Predictions

In [ ]:
_, _, _, a2_final = forward(X, W1, b1, W2, b2)
predictions = np.round(a2_final).astype(int)
print('True y:   ', y.flatten().astype(int))
print('Predicted:', predictions.flatten())
print('Accuracy: ', np.mean(predictions.flatten() == y.flatten()))

## Step 11: Visualize the Network Architecture

Each node represents a neuron. Line color/opacity represents connection strength.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def draw_network(layer_sizes, layer_labels=None, title='', figsize=(9, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.2, 1.2)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, fontweight='bold')
    n = len(layer_sizes)
    xs = np.linspace(0.1, 0.9, n)
    max_n = max(layer_sizes)
    gap = min(0.15, 0.85 / max(max_n, 2))
    colors = ['#74b9ff', '#a29bfe', '#fd79a8', '#55efc4', '#ffeaa7']
    ys_all = []
    for sz in layer_sizes:
        ys = [0.5 + (i - (sz - 1) / 2) * gap for i in range(sz)]
        ys_all.append(ys)
    for li in range(n - 1):
        for ya in ys_all[li]:
            for yb in ys_all[li + 1]:
                ax.plot([xs[li], xs[li + 1]], [ya, yb], color='#dfe6e9', lw=0.8, zorder=1)
    for li, (x, ys) in enumerate(zip(xs, ys_all)):
        for y in ys:
            c = plt.Circle((x, y), 0.038, fc=colors[li % len(colors)], ec='#2d3436', lw=1.2, zorder=5)
            ax.add_patch(c)
        lab = layer_labels[li] if layer_labels else str(layer_sizes[li])
        ax.text(x, min(ys) - 0.1, lab, ha='center', fontsize=9, color='#636e72')
    plt.tight_layout()
    plt.show()

draw_network(
    [2, 4, 1],
    layer_labels=['Input\n(2)', 'Hidden\n(4)', 'Output\n(1)'],
    title='XOR Network Architecture: 2 → 4 → 1'
)

## Step 12: Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(losses, color='#a29bfe', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Training Loss — XOR from Scratch')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()